# 01 — Data Collection

This notebook collects the raw data needed to train both models:
1. **Kaggle MAL Dataset** — anime metadata, synopses, and user ratings
2. **AniList Reviews** — user-written reviews scraped via GraphQL API
3. **MAL↔AniList ID Mapping** — via Fribb's anime-lists

**Runtime**: Kaggle (recommended) or Colab. AniList scraping takes ~1-2 hours due to rate limits.

## Setup

In [1]:
import os
import json
import time
import requests
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path
from collections import defaultdict

# Output directory for all collected data
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"Data directory: {DATA_DIR.resolve()}")

Data directory: C:\Users\jeddh\Projects\animetracker\notebooks\data


## 1. Kaggle MAL Dataset

Download from: https://www.kaggle.com/datasets/hernan4444/anime-recommendation-database-2020

If running on **Kaggle**, add the dataset to your notebook and it will be available at `/kaggle/input/`.

If running on **Colab**, upload the CSV files or use the Kaggle API.

In [2]:
# --- Option A: Running on Kaggle ---
KAGGLE_INPUT = Path("/kaggle/input/anime-recommendation-database-2020")

# --- Option B: Running on Colab (download via Kaggle API) ---
# Uncomment these lines if on Colab:
# !pip install -q kaggle
# # Upload your kaggle.json API key first, then:
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d hernan4444/anime-recommendation-database-2020 -p data/kaggle --unzip
# KAGGLE_INPUT = Path("data/kaggle")

# Detect which path exists
if KAGGLE_INPUT.exists():
    print(f"Using Kaggle input: {KAGGLE_INPUT}")
else:
    KAGGLE_INPUT = DATA_DIR / "kaggle"
    if not KAGGLE_INPUT.exists():
        raise FileNotFoundError(
            "Kaggle dataset not found. Either:\n"
            "  - Add 'anime-recommendation-database-2020' to your Kaggle notebook\n"
            "  - Or uncomment the Colab download lines above"
        )
    print(f"Using local Kaggle data: {KAGGLE_INPUT}")

Using local Kaggle data: data\kaggle


In [3]:
# Load anime metadata
anime_df = pd.read_csv(KAGGLE_INPUT / "anime.csv")
print(f"Anime entries: {len(anime_df):,}")
print(f"Columns: {list(anime_df.columns)}")
anime_df.head()

Anime entries: 17,562
Columns: ['MAL_ID', 'Name', 'Score', 'Genres', 'English name', 'Japanese name', 'Type', 'Episodes', 'Aired', 'Premiered', 'Producers', 'Licensors', 'Studios', 'Source', 'Duration', 'Rating', 'Ranked', 'Popularity', 'Members', 'Favorites', 'Watching', 'Completed', 'On-Hold', 'Dropped', 'Plan to Watch', 'Score-10', 'Score-9', 'Score-8', 'Score-7', 'Score-6', 'Score-5', 'Score-4', 'Score-3', 'Score-2', 'Score-1']


,MAL_ID,Name,Score,Genres,English name,Japanese name,Type,Episodes,Aired,Premiered,...,Score-10,Score-9,Score-8,Score-7,Score-6,Score-5,Score-4,Score-3,Score-2,Score-1
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",Cowboy Bebop,カウボーイビバップ,TV,26,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,...,229170.0,182126.0,131625.0,62330.0,20688.0,8904.0,3184.0,1357.0,741.0,1580.0
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space",Cowboy Bebop:The Movie,カウボーイビバップ 天国の扉,Movie,1,"Sep 1, 2001",Unknown,...,30043.0,49201.0,49505.0,22632.0,5805.0,1877.0,577.0,221.0,109.0,379.0
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen",Trigun,トライガン,TV,26,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,...,50229.0,75651.0,86142.0,49432.0,15376.0,5838.0,1965.0,664.0,316.0,533.0
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",Witch Hunter Robin,Witch Hunter ROBIN (ウイッチハンターロビン),TV,26,"Jul 2, 2002 to Dec 24, 2002",Summer 2002,...,2182.0,4806.0,10128.0,11618.0,5709.0,2920.0,1083.0,353.0,164.0,131.0
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",Beet the Vandel Buster,冒険王ビィト,TV,52,"Sep 30, 2004 to Sep 29, 2005",Fall 2004,...,312.0,529.0,1242.0,1713.0,1068.0,634.0,265.0,83.0,50.0,27.0


In [4]:
# Load synopses
synopsis_df = pd.read_csv(KAGGLE_INPUT / "anime_with_synopsis.csv")
print(f"Anime with synopses: {len(synopsis_df):,}")
synopsis_df.head()

Anime with synopses: 16,214


,MAL_ID,Name,Score,Genres,sypnopsis
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space","In the year 2071, humanity has colonized sever..."
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space","other day, another bounty—such is the life of ..."
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen","Vash the Stampede is the man with a $$60,000,0..."
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",ches are individuals with special powers like ...
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",It is the dark century and the people are suff...


In [5]:
# Load user ratings
ratings_df = pd.read_csv(KAGGLE_INPUT / "rating_complete.csv")
print(f"Total ratings: {len(ratings_df):,}")
print(f"Unique users: {ratings_df['user_id'].nunique():,}")
print(f"Unique anime: {ratings_df['anime_id'].nunique():,}")
ratings_df.head()

Total ratings: 57,633,278
Unique users: 310,059
Unique anime: 16,872


,user_id,anime_id,rating
0,0,430,9
1,0,1004,5
2,0,3010,7
3,0,570,7
4,0,2762,9


In [6]:
# Filter: users with 20+ ratings, anime with 50+ ratings
MIN_USER_RATINGS = 20
MIN_ANIME_RATINGS = 50

# Count ratings per user and per anime
user_counts = ratings_df['user_id'].value_counts()
anime_counts = ratings_df['anime_id'].value_counts()

# Filter
active_users = user_counts[user_counts >= MIN_USER_RATINGS].index
popular_anime = anime_counts[anime_counts >= MIN_ANIME_RATINGS].index

filtered_ratings = ratings_df[
    ratings_df['user_id'].isin(active_users) &
    ratings_df['anime_id'].isin(popular_anime)
].copy()

print(f"\nAfter filtering:")
print(f"  Ratings: {len(filtered_ratings):,} (was {len(ratings_df):,})")
print(f"  Users: {filtered_ratings['user_id'].nunique():,} (was {ratings_df['user_id'].nunique():,})")
print(f"  Anime: {filtered_ratings['anime_id'].nunique():,} (was {ratings_df['anime_id'].nunique():,})")


After filtering:
  Ratings: 57,178,041 (was 57,633,278)
  Users: 265,263 (was 310,059)
  Anime: 12,127 (was 16,872)


In [7]:
# Save filtered ratings
filtered_ratings.to_csv(DATA_DIR / "ratings_filtered.csv", index=False)
print(f"Saved filtered ratings to {DATA_DIR / 'ratings_filtered.csv'}")

Saved filtered ratings to data\ratings_filtered.csv


## 2. MAL↔AniList ID Mapping

We use [Fribb's anime-lists](https://github.com/Fribb/anime-lists) which maintains a community-curated mapping between MAL, AniList, Kitsu, and other anime databases.

In [8]:
# Download Fribb's anime-lists mapping
FRIBB_URL = "https://raw.githubusercontent.com/Fribb/anime-lists/master/anime-list-full.json"

print("Downloading Fribb's anime-lists mapping...")
resp = requests.get(FRIBB_URL, timeout=60)
resp.raise_for_status()
fribb_data = resp.json()
print(f"Total entries in Fribb's list: {len(fribb_data):,}")

Total entries in Fribb's list: 41,808


In [9]:
# Build MAL ID <-> AniList ID mapping
mal_to_anilist = {}
anilist_to_mal = {}

for entry in tqdm(fribb_data, desc="Building ID map"):
    mal_id = entry.get("mal_id")
    anilist_id = entry.get("anilist_id")
    if mal_id and anilist_id:
        mal_to_anilist[int(mal_id)] = int(anilist_id)
        anilist_to_mal[int(anilist_id)] = int(mal_id)

print(f"Mapped entries: {len(mal_to_anilist):,}")

# Check coverage against our filtered anime
filtered_mal_ids = set(filtered_ratings['anime_id'].unique())
mapped_mal_ids = set(mal_to_anilist.keys())
covered = filtered_mal_ids & mapped_mal_ids
print(f"Coverage: {len(covered):,}/{len(filtered_mal_ids):,} filtered anime have AniList IDs ({100*len(covered)/len(filtered_mal_ids):.1f}%)")

Building ID map:   0%|          | 0/41808 [00:00<?, ?it/s]

Mapped entries: 18,622
Coverage: 9,988/12,127 filtered anime have AniList IDs (82.4%)


In [10]:
# Save ID mapping
id_map = {
    "mal_to_anilist": {str(k): v for k, v in mal_to_anilist.items()},
    "anilist_to_mal": {str(k): v for k, v in anilist_to_mal.items()}
}

with open(DATA_DIR / "id_map.json", "w") as f:
    json.dump(id_map, f)

print(f"Saved ID mapping to {DATA_DIR / 'id_map.json'}")

Saved ID mapping to data\id_map.json


## 3. AniList Review Scraping

Scrape reviews from AniList's GraphQL API for the top anime by popularity.

**Rate limit**: AniList allows 90 requests/minute. We add a 0.7s delay between requests to stay safe.

In [11]:
ANILIST_API = "https://graphql.anilist.co"

# GraphQL query to get popular anime IDs
POPULAR_ANIME_QUERY = """
query ($page: Int, $perPage: Int) {
  Page(page: $page, perPage: $perPage) {
    pageInfo {
      hasNextPage
      currentPage
    }
    media(type: ANIME, sort: POPULARITY_DESC) {
      id
      idMal
      title { romaji english }
      genres
      tags { name rank }
      popularity
      averageScore
    }
  }
}
"""

# GraphQL query to get reviews for a specific anime
REVIEWS_QUERY = """
query ($mediaId: Int, $page: Int, $perPage: Int) {
  Page(page: $page, perPage: $perPage) {
    pageInfo {
      hasNextPage
      currentPage
    }
    reviews(mediaId: $mediaId, sort: RATING_DESC) {
      id
      body
      score
      rating
      ratingAmount
    }
  }
}
"""

In [12]:
def anilist_request(query, variables, max_retries=3):
    """Make a rate-limited request to AniList GraphQL API."""
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                ANILIST_API,
                json={"query": query, "variables": variables},
                timeout=30
            )
            if resp.status_code == 429:
                retry_after = int(resp.headers.get("Retry-After", 60))
                print(f"  Rate limited, waiting {retry_after}s...")
                time.sleep(retry_after)
                continue
            resp.raise_for_status()
            return resp.json()["data"]
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                time.sleep(5)
                continue
            raise
    return None

In [13]:
# Step 1: Fetch top 5000 anime by popularity
TARGET_ANIME = 5000
PER_PAGE = 50

anime_list = []
page = 1

print(f"Fetching top {TARGET_ANIME} anime by popularity...")
pbar = tqdm(total=TARGET_ANIME, desc="Fetching anime")
while len(anime_list) < TARGET_ANIME:
    data = anilist_request(POPULAR_ANIME_QUERY, {"page": page, "perPage": PER_PAGE})
    if not data or not data["Page"]["media"]:
        break

    before_count = len(anime_list)
    for media in data["Page"]["media"]:
        anime_list.append({
            "anilist_id": media["id"],
            "mal_id": media.get("idMal"),
            "title": media["title"].get("romaji") or media["title"].get("english", "Unknown"),
            "genres": media.get("genres", []),
            "tags": [{"name": t["name"], "rank": t["rank"]} for t in (media.get("tags") or [])],
            "popularity": media.get("popularity", 0),
            "average_score": media.get("averageScore")
        })

    pbar.update(len(anime_list) - before_count)

    has_next = data["Page"]["pageInfo"]["hasNextPage"]
    if not has_next:
        break

    page += 1
    time.sleep(0.7)  # Rate limit

    if len(anime_list) % 500 == 0:
        print(f"  Fetched {len(anime_list)} anime...")

pbar.close()
print(f"\nTotal anime fetched: {len(anime_list):,}")

Fetching top 5000 anime by popularity...


Fetching anime:   0%|          | 0/5000 [00:00<?, ?it/s]

  Fetched 500 anime...
  Rate limited, waiting 60s...
  Fetched 1000 anime...
  Fetched 1500 anime...
  Rate limited, waiting 60s...
  Fetched 2000 anime...
  Rate limited, waiting 60s...
  Fetched 2500 anime...
  Fetched 3000 anime...
  Rate limited, waiting 60s...
  Fetched 3500 anime...
  Fetched 4000 anime...
  Rate limited, waiting 60s...
  Fetched 4500 anime...
  Rate limited, waiting 60s...
  Fetched 5000 anime...

Total anime fetched: 5,000


In [14]:
# Save anime metadata (useful for corpus building later)
with open(DATA_DIR / "anilist_anime.jsonl", "w") as f:
    for anime in tqdm(anime_list, desc="Saving anime metadata"):
        f.write(json.dumps(anime) + "\n")

print(f"Saved {len(anime_list)} anime metadata entries")

Saving anime metadata:   0%|          | 0/5000 [00:00<?, ?it/s]

Saved 5000 anime metadata entries


In [15]:
# Step 2: Scrape reviews for each anime
# This is the slow part — ~1-2 hours for 5000 anime

REVIEWS_PER_ANIME = 25  # Max reviews to fetch per anime
REVIEW_PER_PAGE = 25

# Resume support: check if we already have partial results
reviews_file = DATA_DIR / "anilist_reviews.jsonl"
scraped_ids = set()
if reviews_file.exists():
    with open(reviews_file, "r") as f:
        for line in f:
            review = json.loads(line)
            scraped_ids.add(review["anilist_id"])
    print(f"Resuming: {len(scraped_ids)} anime already scraped")

remaining = [a for a in anime_list if a["anilist_id"] not in scraped_ids]
print(f"Anime to scrape: {len(remaining)}")

Anime to scrape: 5000


In [16]:
# Scrape reviews (append mode for resume support)
total_reviews = 0
anime_with_reviews = 0

with open(reviews_file, "a") as f:
    for i, anime in enumerate(tqdm(remaining, desc="Scraping reviews")):
        anilist_id = anime["anilist_id"]
        mal_id = anime.get("mal_id")

        # Fetch first page of reviews
        data = anilist_request(REVIEWS_QUERY, {
            "mediaId": anilist_id,
            "page": 1,
            "perPage": REVIEW_PER_PAGE
        })

        if not data or not data["Page"]["reviews"]:
            time.sleep(0.7)
            continue

        anime_with_reviews += 1
        for review in data["Page"]["reviews"]:
            body = review.get("body", "")
            if not body or len(body) < 100:  # Skip very short reviews
                continue

            entry = {
                "anilist_id": anilist_id,
                "mal_id": mal_id,
                "review_id": review["id"],
                "body": body,
                "score": review.get("score"),
                "rating": review.get("rating", 0),
                "rating_amount": review.get("ratingAmount", 0)
            }
            f.write(json.dumps(entry) + "\n")
            total_reviews += 1

        time.sleep(0.7)  # Rate limit

        if (i + 1) % 100 == 0:
            print(f"  Progress: {i+1}/{len(remaining)} anime, {total_reviews} reviews collected")

print(f"\nDone! Collected {total_reviews} reviews from {anime_with_reviews} anime")

Scraping reviews:   0%|          | 0/5000 [00:00<?, ?it/s]

  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Progress: 100/5000 anime, 1755 reviews collected
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Progress: 200/5000 anime, 3099 reviews collected
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Progress: 300/5000 anime, 4040 reviews collected
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Rate limited, waiting 60s...
  Progress: 400/5000 anime, 4878 reviews collected
  Rate limited, waiting 60s...
  Rate limited, waiti

## 4. Data Summary

In [17]:
# Summary of collected data
print("=" * 50)
print("DATA COLLECTION SUMMARY")
print("=" * 50)

# Ratings
print(f"\n[Ratings]")
print(f"  Total filtered ratings: {len(filtered_ratings):,}")
print(f"  Unique users: {filtered_ratings['user_id'].nunique():,}")
print(f"  Unique anime (MAL): {filtered_ratings['anime_id'].nunique():,}")

# ID Mapping
print(f"\n[ID Mapping]")
print(f"  MAL→AniList entries: {len(mal_to_anilist):,}")

# Reviews
review_count = 0
review_anime_ids = set()
if reviews_file.exists():
    with open(reviews_file, "r") as f:
        for line in f:
            review = json.loads(line)
            review_count += 1
            review_anime_ids.add(review["anilist_id"])

print(f"\n[Reviews]")
print(f"  Total reviews: {review_count:,}")
print(f"  Anime with reviews: {len(review_anime_ids):,}")
if review_count > 0:
    print(f"  Avg reviews/anime: {review_count / len(review_anime_ids):.1f}")

# Anime metadata
print(f"\n[Anime Metadata]")
print(f"  AniList anime scraped: {len(anime_list):,}")
print(f"  Kaggle anime with synopses: {len(synopsis_df):,}")

print(f"\n[Output Files]")
for f_path in sorted(DATA_DIR.glob("*")):
    size_mb = f_path.stat().st_size / (1024 * 1024)
    print(f"  {f_path.name}: {size_mb:.1f} MB")

print("\n✓ Data collection complete. Proceed to 02_preprocessing.ipynb")

DATA COLLECTION SUMMARY

[Ratings]
  Total filtered ratings: 57,178,041
  Unique users: 265,263
  Unique anime (MAL): 12,127

[ID Mapping]
  MAL→AniList entries: 18,622

[Reviews]
  Total reviews: 12,481
  Anime with reviews: 2,797
  Avg reviews/anime: 4.5

[Anime Metadata]
  AniList anime scraped: 5,000
  Kaggle anime with synopses: 16,214

[Output Files]
  anilist_anime.jsonl: 3.3 MB
  anilist_reviews.jsonl: 80.4 MB
  id_map.json: 0.6 MB
  kaggle: 0.0 MB
  ratings_filtered.csv: 828.4 MB

✓ Data collection complete. Proceed to 02_preprocessing.ipynb
